Week 4 Week 4-5 Deliverable: Data Cleaning and Preparation

1. Converts date fields to datetime format
2. Removes unnecessary/redundant columns (>90% null, from Week 2-3 report)
3. Ensures numeric fields are properly typed
4. Flags invalid numeric values (ClosePrice<=0, LivingArea<=0, DaysOnMarket<0,
   negative Bedrooms/Bathrooms)
5. Runs date consistency checks (listing_after_close_flag,
   purchase_after_close_flag, negative_timeline_flag)
6. Runs geographic data checks (missing/sentinel/wrong-sign coordinates)
7. Saves a cleaned, analysis-ready CSV for each dataset

In [22]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)
 
DATE_FIELDS = ['CloseDate', 'PurchaseContractDate', 'ListingContractDate', 'ContractStatusChangeDate']
 

In [23]:
# Columns with >90% missing values found in the Week 2-3 EDA report.

HIGH_NULL_COLUMNS_TO_DROP = [
    'MiddleOrJuniorSchoolDistrict',
    'BusinessType',
    'TaxYear',
    'ElementarySchoolDistrict',
    'TaxAnnualAmount',
    'FireplacesTotal',
    'AboveGradeFinishedArea',
]
 
NUMERIC_FIELDS = ['ClosePrice', 'ListPrice', 'OriginalListPrice', 'LivingArea',
                   'LotSizeAcres', 'BedroomsTotal', 'BathroomsTotalInteger',
                   'DaysOnMarket', 'YearBuilt']

In [10]:
# PART 1 - SOLD DATASET
# =================================================================
print("=" * 70)
print("CLEANING: SOLD  (sold_with_rates.csv)")
print("=" * 70)
df_sold = pd.read_csv('sold_with_rates.csv', low_memory=False)
rows_before_sold = len(df_sold)
cols_before_sold = len(df_sold.columns)
print(f"\nLoaded {rows_before_sold} rows, {cols_before_sold} columns")

CLEANING: SOLD  (sold_with_rates.csv)

Loaded 448091 rows, 84 columns


In [24]:
df_sold.head()

,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,PropertyType,LivingArea,ListPrice,DaysOnMarket,ListOfficeName,BuyerOfficeName,CoListOfficeName,ListAgentFullName,CoListAgentFirstName,CoListAgentLastName,BuyerAgentMlsId,BuyerAgentFirstName,BuyerAgentLastName,AssociationFeeFrequency,ListingKeyNumeric,MLSAreaMajor,CountyOrParish,MlsStatus,ElementarySchool,AttachedGarageYN,ParkingTotal,BuilderName,PropertySubType,LotSizeAcres,SubdivisionName,BuyerOfficeAOR,YearBuilt,StreetNumberNumeric,ListingId,BathroomsTotalInteger,City,BuildingAreaTotal,BedroomsTotal,ContractStatusChangeDate,CoBuyerAgentFirstName,PurchaseContractDate,ListingContractDate,BelowGradeFinishedArea,StateOrProvince,CoveredSpaces,MiddleOrJuniorSchool,FireplaceYN,Stories,HighSchool,Levels,LotSizeDimensions,LotSizeArea,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,OriginatingSystemName,OriginatingSystemSubName,BuyerAgencyCompensationType,BuyerAgencyCompensation,year_month,rate_30yr_fixed,invalid_close_price_flag,invalid_living_area_flag,invalid_days_on_market_flag,invalid_bedrooms_flag,invalid_bathrooms_flag,listing_after_close_flag,purchase_after_close_flag,negative_timeline_flag,missing_coords_flag,sentinel_zero_coords_flag,wrong_sign_longitude_flag,implausible_coords_flag
0,Mlslistings,Mlslistings,"Carpet,Tile,Wood",True,NaN,NaN,False,499000.0,551985747,jwachter@cbnorcal.com,2024-01-26,240000.0,Joan,Wachter,NaN,NaN,1 Baldwin Avenue 411,Residential,1140.0,295000.0,777,Coldwell Banker Realty,Bay Area Senior Services,NaN,Joan Wachter,NaN,NaN,ML5090968,Scott,Withrow,Monthly,551985747,699 - Not Defined,San Mateo,Closed,NaN,False,1.0,NaN,Condominium,NaN,NaN,Mlslistings,1988.0,1.0,ML81865679,2.0,San Mateo,NaN,2.0,2024-01-26,NaN,2023-11-22,2021-10-06,NaN,CA,NaN,NaN,True,NaN,NaN,NaN,NaN,NaN,NaN,False,1.0,Other,94401,6472.0,NaN,CRMLS,CRMLS_MLSL,NaN,NaN,2024-01,6.6425,False,False,False,False,False,False,False,False,True,False,False,False
1,SanDiego,SanDiego,NaN,False,NaN,NaN,False,759900.0,522107581,mdarwich12@gmail.com,2024-01-05,815000.0,Michael,Darwich,NaN,NaN,2811 C Avenue,Residential,1974.0,759900.0,33,Berkshire Hathaway HomeServices California Pro...,FOSTER HAMILTON Real Estate,NaN,Michael Darwich,NaN,NaN,SAND-606727,Charles,Street,NaN,522107581,91950 - National City,San Diego,Closed,NaN,True,2.0,NaN,SingleFamilyResidence,NaN,National City,SanDiego,2023.0,2811.0,210014555,4.0,National City,NaN,4.0,2024-01-05,NaN,2021-06-30,2021-03-08,NaN,CA,NaN,NaN,False,2.0,NaN,Two,NaN,NaN,NaN,False,2.0,NaN,91950,NaN,NaN,CRMLS,CRMLS_SAND,NaN,NaN,2024-01,6.6425,False,False,False,False,False,False,False,False,True,False,False,False
2,SanDiego,SanDiego,NaN,False,NaN,NaN,False,739900.0,510919001,mdarwich12@gmail.com,2024-01-05,810000.0,Michael,Darwich,NaN,NaN,2812 C Avenue,Residential,1974.0,770000.0,228,Berkshire Hathaway HomeServices California Pro...,FOSTER HAMILTON Real Estate,NaN,Michael Darwich,NaN,NaN,SAND-606727,Charles,Street,NaN,510919001,91950 - National City,San Diego,Closed,NaN,True,2.0,NaN,SingleFamilyResidence,NaN,National City,SanDiego,2023.0,2812.0,210008330,4.0,National City,NaN,4.0,2024-01-05,NaN,2021-11-18,2021-03-08,NaN,CA,NaN,NaN,False,2.0,NaN,Two,NaN,NaN,NaN,False,2.0,NaN,91950,NaN,NaN,CRMLS,CRMLS_SAND,NaN,NaN,2024-01,6.6425,False,False,False,False,False,False,False,False,True,False,False,False
3,Mlslistings,Mlslistings,NaN,False,NaN,NaN,NaN,NaN,1079166779,davidmartz@compass.com,2024-01-30,858000.0,David,Martz,NaN,NaN,2199 N Berne Drive,Residential,1995.0,858000.0,0,Compass,NaN,NaN,David Martz,NaN,NaN,ML921886,Steven,Hill,NaN,1079166779,NaN,Riverside,Closed,Vista Del Monte,False,2.0,NaN,SingleFamilyResidence,0.3100,NaN,NaN,1959.0,2199.0,ML81975626,3.0,Palm Springs,NaN,0.0,2024-01-30,NaN,2024-08-05,2024-01-30,NaN,CA,NaN,Raymond Cree,False,

In [25]:
# --- 1. Convert date fields to datetime ---
df_sold['CloseDate'] = pd.to_datetime(df_sold['CloseDate'], errors='coerce')
df_sold['PurchaseContractDate'] = pd.to_datetime(df_sold['PurchaseContractDate'], errors='coerce')
df_sold['ListingContractDate'] = pd.to_datetime(df_sold['ListingContractDate'], errors='coerce')
df_sold['ContractStatusChangeDate'] = pd.to_datetime(df_sold['ContractStatusChangeDate'], errors='coerce') 

In [26]:
# --- 2. Remove high-null / redundant columns ---
print("\n--- 2. Removing high-null / redundant columns ---")
cols_present_sold = [c for c in HIGH_NULL_COLUMNS_TO_DROP if c in df_sold.columns]
cols_before_drop_sold = len(df_sold.columns)
df_sold = df_sold.drop(columns=cols_present_sold)

print(f"  Dropped {len(cols_present_sold)} columns: {cols_present_sold}")
print(f"  Columns before: {cols_before_drop_sold}, after: {len(df_sold.columns)}")


--- 2. Removing high-null / redundant columns ---
  Dropped 0 columns: []
  Columns before: 89, after: 89


In [27]:
# --- 3. Ensure numeric fields are properly typed ---
print("\n--- 3. Ensuring numeric fields are properly typed ---")
for col in NUMERIC_FIELDS:
    if col in df_sold.columns:
        before_dtype = df_sold[col].dtype
        df_sold[col] = pd.to_numeric(df_sold[col], errors='coerce')
        print(f"  {col}: {before_dtype} -> {df_sold[col].dtype}")


--- 3. Ensuring numeric fields are properly typed ---
  ClosePrice: float64 -> float64
  ListPrice: float64 -> float64
  OriginalListPrice: float64 -> float64
  LivingArea: float64 -> float64
  LotSizeAcres: float64 -> float64
  BedroomsTotal: float64 -> float64
  BathroomsTotalInteger: float64 -> float64
  DaysOnMarket: int64 -> int64
  YearBuilt: float64 -> float64


In [28]:
# --- 4. Flag invalid numeric values ---
print("\n--- 4. Flagging invalid numeric values ---")
df_sold['invalid_close_price_flag'] = df_sold['ClosePrice'] <= 0 if 'ClosePrice' in df_sold.columns else False
df_sold['invalid_living_area_flag'] = df_sold['LivingArea'] <= 0 if 'LivingArea' in df_sold.columns else False
df_sold['invalid_days_on_market_flag'] = df_sold['DaysOnMarket'] < 0 if 'DaysOnMarket' in df_sold.columns else False
df_sold['invalid_bedrooms_flag'] = df_sold['BedroomsTotal'] < 0 if 'BedroomsTotal' in df_sold.columns else False
df_sold['invalid_bathrooms_flag'] = df_sold['BathroomsTotalInteger'] < 0 if 'BathroomsTotalInteger' in df_sold.columns else False
 
print(f"  invalid_close_price_flag:    {df_sold['invalid_close_price_flag'].sum()} records")
print(f"  invalid_living_area_flag:    {df_sold['invalid_living_area_flag'].sum()} records")
print(f"  invalid_days_on_market_flag: {df_sold['invalid_days_on_market_flag'].sum()} records")
print(f"  invalid_bedrooms_flag:       {df_sold['invalid_bedrooms_flag'].sum()} records")
print(f"  invalid_bathrooms_flag:      {df_sold['invalid_bathrooms_flag'].sum()} records")


--- 4. Flagging invalid numeric values ---
  invalid_close_price_flag:    1 records
  invalid_living_area_flag:    165 records
  invalid_days_on_market_flag: 50 records
  invalid_bedrooms_flag:       0 records
  invalid_bathrooms_flag:      0 records


In [29]:
# --- 5. Date consistency checks ---
print("\n--- 5. Date consistency checks ---")
has_all_date_fields_sold = all(c in df_sold.columns for c in ['ListingContractDate', 'PurchaseContractDate', 'CloseDate'])
 
if has_all_date_fields_sold:
    df_sold['listing_after_close_flag'] = df_sold['ListingContractDate'] > df_sold['CloseDate']
    df_sold['purchase_after_close_flag'] = df_sold['PurchaseContractDate'] > df_sold['CloseDate']
    df_sold['negative_timeline_flag'] = (
        df_sold['listing_after_close_flag']
        | df_sold['purchase_after_close_flag']
        | (df_sold['ListingContractDate'] > df_sold['PurchaseContractDate'])
    )
 
    print(f"  listing_after_close_flag:  {df_sold['listing_after_close_flag'].sum()} records "
          f"(ListingContractDate is after CloseDate)")
    print(f"  purchase_after_close_flag: {df_sold['purchase_after_close_flag'].sum()} records "
          f"(PurchaseContractDate is after CloseDate)")
    print(f"  negative_timeline_flag:    {df_sold['negative_timeline_flag'].sum()} records "
          f"(any date-order violation)")
else:
    print("  Missing one or more of ListingContractDate/PurchaseContractDate/CloseDate, skipping.")
    df_sold['listing_after_close_flag'] = False
    df_sold['purchase_after_close_flag'] = False
    df_sold['negative_timeline_flag'] = False


--- 5. Date consistency checks ---
  listing_after_close_flag:  68 records (ListingContractDate is after CloseDate)
  purchase_after_close_flag: 241 records (PurchaseContractDate is after CloseDate)
  negative_timeline_flag:    531 records (any date-order violation)


In [30]:
# --- 6. Geographic data checks ---
print("\n--- 6. Geographic data checks ---")
has_coords_sold = 'Latitude' in df_sold.columns and 'Longitude' in df_sold.columns
 
if has_coords_sold:
    df_sold['Latitude'] = pd.to_numeric(df_sold['Latitude'], errors='coerce')
    df_sold['Longitude'] = pd.to_numeric(df_sold['Longitude'], errors='coerce')
 
    df_sold['missing_coords_flag'] = df_sold['Latitude'].isnull() | df_sold['Longitude'].isnull()
    df_sold['sentinel_zero_coords_flag'] = (df_sold['Latitude'] == 0) | (df_sold['Longitude'] == 0)
    df_sold['wrong_sign_longitude_flag'] = df_sold['Longitude'] > 0
 
    df_sold['implausible_coords_flag'] = (
        ~df_sold['missing_coords_flag'] &
        ((df_sold['Latitude'] < 32) | (df_sold['Latitude'] > 42) |
         (df_sold['Longitude'] < -125) | (df_sold['Longitude'] > -114))
    )
 
    print(f"  missing_coords_flag:        {df_sold['missing_coords_flag'].sum()} records")
    print(f"  sentinel_zero_coords_flag:  {df_sold['sentinel_zero_coords_flag'].sum()} records")
    print(f"  wrong_sign_longitude_flag:  {df_sold['wrong_sign_longitude_flag'].sum()} records")
    print(f"  implausible_coords_flag:    {df_sold['implausible_coords_flag'].sum()} records "
          f"(outside rough CA lat/lon bounding box)")
else:
    print("  Latitude/Longitude not present, skipping.")
    df_sold['missing_coords_flag'] = False
    df_sold['sentinel_zero_coords_flag'] = False
    df_sold['wrong_sign_longitude_flag'] = False
    df_sold['implausible_coords_flag'] = False
 


--- 6. Geographic data checks ---
  missing_coords_flag:        16222 records
  sentinel_zero_coords_flag:  37 records
  wrong_sign_longitude_flag:  31 records
  implausible_coords_flag:    99 records (outside rough CA lat/lon bounding box)


In [38]:
# --- 7. Add school district mapping (Aidan's Slack task) ---
print("\n--- 7. Adding school district mapping ---")

import geopandas as gpd
from shapely.geometry import Point

GEOJSON_PATH = 'california_school_district_areas_2025_26.geojson'

districts = gpd.read_file(GEOJSON_PATH)
districts_unified = districts[districts['DistrictType'] == 'Unified'].copy()
print(f"  Loaded {len(districts)} district polygons, filtered to {len(districts_unified)} Unified districts")

join_mask_sold = df_sold['Latitude'].notnull() & df_sold['Longitude'].notnull()
df_for_join_sold = df_sold[join_mask_sold].copy()

geometry_sold = [Point(lon, lat) for lon, lat in
                  zip(df_for_join_sold['Longitude'], df_for_join_sold['Latitude'])]
gdf_sold = gpd.GeoDataFrame(df_for_join_sold, geometry=geometry_sold, crs='EPSG:4326')

if districts_unified.crs != gdf_sold.crs:
    districts_unified = districts_unified.to_crs(gdf_sold.crs)

joined_sold = gpd.sjoin(
    gdf_sold,
    districts_unified[['DistrictName', 'geometry']],
    how='left',
    predicate='within'
)

df_sold = df_sold.merge(joined_sold[['DistrictName']], left_index=True, right_index=True, how='left')

print(f"  Matched to a Unified school district: {df_sold['DistrictName'].notnull().sum()} records")
print(f"  Not matched (missing coords or outside any polygon): {df_sold['DistrictName'].isnull().sum()} records")


--- 7. Adding school district mapping ---
  Loaded 936 district polygons, filtered to 345 Unified districts
  Matched to a Unified school district: 327953 records
  Not matched (missing coords or outside any polygon): 120138 records


In [39]:
# --- 8. Summary + save ---
rows_after_sold = len(df_sold)
cols_after_sold = len(df_sold.columns)
 
print(f"\n--- Summary for sold ---")
print(f"Rows before: {rows_before_sold}, Rows after: {rows_after_sold} (no rows dropped, only flagged/enriched)")
print(f"Columns before: {cols_before_sold}, Columns after: {cols_after_sold}")
 
df_sold.to_csv('sold_cleaned.csv', index=False, encoding='utf-8')
print(f"\nSaved cleaned dataset: sold_cleaned.csv ({rows_after_sold} rows, {cols_after_sold} columns)")


--- Summary for sold ---
Rows before: 448091, Rows after: 448091 (no rows dropped, only flagged/enriched)
Columns before: 84, Columns after: 90

Saved cleaned dataset: sold_cleaned.csv (448091 rows, 90 columns)
